# Editing semiconductor defect images with Boogu-Image-0.1-Edit-Turbo\n\nBoogu-Image-0.1 ([GitHub](https://github.com/boogu-project/Boogu-Image), [Boogu/Boogu-Image-0.1-Edit-Turbo on HuggingFace](https://huggingface.co/Boogu/Boogu-Image-0.1-Edit-Turbo)) is a 10B-parameter, Apache-2.0-licensed, unified image generation/editing model released in mid-2026. It reportedly matches or beats larger closed models (e.g. Qwen-Image-2512 at 20B, Hunyuan-Image-3.0 at 80B) on public benchmarks despite roughly an order of magnitude less training data. The `Edit-Turbo` variant is a 4-step distilled checkpoint for instruction-guided image editing: give it an image and a text instruction, get back an edited image.\n\nThis notebook is a first look at using it to edit semiconductor wafer/die defect images -- the kind of thing that shows up in fab process-control work (crack, bridge, scratch, via, short, open defects on SEM/optical inspection images). The goal: see whether a general-purpose instruction-following edit model can do useful things with defect imagery -- annotate/highlight a defect region, simulate a defect on a clean image, or restyle an image for a figure -- without any fine-tuning.\n\n**On the source images:** the demo image used here (`assets/boogu-image-demo/wafer_scratch_defect.png`) is a **synthetically generated** wafer-pattern image, not a real fab photo or a scan of a journal figure. I picked it deliberately: the Kaggle dataset it came from (`surjini/two-stage-semiconductor-defect-dataset-sem-img`) turned out to be a mix of at least three kinds of images -- procedurally generated synthetic patterns (safe to reuse), scanned figures from paywalled journal articles (filenames like `1-s2.0-<pii>-grN.jpg` -- these are ScienceDirect's own figure-export naming scheme, i.e. copyrighted paper figures), and images that look scraped from a Bing image search (`OIP*.jpg`, `OIP (n).jpg` -- Bing's thumbnail cache naming convention, unknown provenance/license). **Don't publish anything built from the journal-figure or OIP-named files without checking the actual copyright first** -- a dataset's Kaggle license badge only covers what the uploader actually had the rights to grant, and image search results and paper figures usually aren't public domain. The synthetic-pattern files (`*_defect_NNN.png` naming) are the safe subset."]

## Setup\n\nRun from a dedicated venv (`.venv-boogu`) rather than the blog's main environment, since this pulls in a heavy, GPU-specific stack (torch/diffusers/transformers/accelerate) that has nothing to do with building the Quarto site:\n\n```bash\ncd quarto_blog_hasan\nuv venv --python 3.12 .venv-boogu\nsource .venv-boogu/bin/activate\nuv pip install torch diffusers transformers accelerate huggingface_hub pillow ipykernel\npython -m ipykernel install --user --name boogu --display-name \"Python 3 (boogu)\"\n```\n\nWeights are cached under `.model_cache/` (gitignored -- it's ~39GB, and `Boogu/Boogu-Image-0.1-Edit-Turbo` on HuggingFace is the canonical source, no reason to duplicate it into version control).

In [ ]:
import torch\nfrom diffusers import DiffusionPipeline\nfrom diffusers.utils import load_image\nfrom pathlib import Path\nimport matplotlib.pyplot as plt\n\n# Paths are relative to this notebook's own directory (quarto_blog_hasan/notebooks/).\nASSETS = Path(\"assets/boogu-image-demo\")\nMODEL_CACHE = Path(\"/home/hasan-spark/Models/boogu-image-0.1-edit-turbo_cache\")\nOUTPUT_DIR = ASSETS / \"outputs\"\nOUTPUT_DIR.mkdir(parents=True, exist_ok=True)\n\nprint(\"torch\", torch.__version__, \"| CUDA available:\", torch.cuda.is_available())\nif torch.cuda.is_available():\n    print(\"device:\", torch.cuda.get_device_name(0))

## Load the model\n\nThe unified `DiffusionPipeline.from_pretrained(...)` entry point handles picking the right pipeline class for this checkpoint automatically (per the [model card](https://huggingface.co/Boogu/Boogu-Image-0.1-Edit-Turbo)). Loaded in `bfloat16`; this is a 10B-parameter model, so expect ~20GB+ of memory just for weights -- watch `nvidia-smi` / system memory if you're on a machine running much else at the same time (unified-memory chips like this one's GB10 share RAM between CPU and GPU, so other running apps compete directly with the model for the same pool).

In [ ]:
pipe = DiffusionPipeline.from_pretrained(\n    \"Boogu/Boogu-Image-0.1-Edit-Turbo\",\n    cache_dir=MODEL_CACHE,\n    dtype=torch.bfloat16,\n    device_map=\"cuda\",\n)

## The input image\n\nA synthetically generated wafer-scratch-pattern image (see the licensing note above for why this one specifically, and not one of the journal/web-scraped files in the same dataset).

In [ ]:
input_image = load_image(str(ASSETS / \"wafer_scratch_defect.png\"))\nprint(input_image.size)\nplt.figure(figsize=(4, 4))\nplt.imshow(input_image)\nplt.axis(\"off\")\nplt.title(\"input: wafer_scratch_defect.png\")\nplt.show()

## Try a few edit instructions\n\nThree instructions that map to real inspection-workflow tasks: **highlighting** a defect for a report, **annotating** it for a figure, and a **stress test** (asking the model to remove the defect entirely) that's useful for gauging whether the model actually understands what a defect *is* in this image versus just following surface-level instructions. Default inference settings from the model card (4 steps, guidance scale baked into the turbo checkpoint's config) -- no extra tuning.

In [ ]:
prompts = {\n    \"highlight_red\": \"Highlight the scratch defect in this wafer image by outlining it in bright red, keep everything else unchanged\",\n    \"annotate_arrow\": \"Add a clear white arrow and the label 'DEFECT' pointing at the scratch defect in this wafer pattern image\",\n    \"remove_defect\": \"Remove the scratch defect from this wafer pattern image so it looks like a clean, defect-free wafer\",\n}\n\nresults = {}\nfor name, prompt in prompts.items():\n    print(f\"running: {name!r} -> {prompt!r}\")\n    results[name] = pipe(image=input_image, prompt=prompt).images[0]\n    out_path = OUTPUT_DIR / f\"{name}.png\"\n    results[name].save(out_path)\n    print(f\"  saved to {out_path}\")

## Compare input vs. each edit

In [ ]:
fig, axes = plt.subplots(1, len(results) + 1, figsize=(4 * (len(results) + 1), 4))\n\naxes[0].imshow(input_image)\naxes[0].set_title(\"input\")\naxes[0].axis(\"off\")\n\nfor ax, (name, img) in zip(axes[1:], results.items()):\n    ax.imshow(img)\n    ax.set_title(name)\n    ax.axis(\"off\")\n\nplt.tight_layout()\ncomparison_path = OUTPUT_DIR / \"comparison_grid.png\"\nplt.savefig(comparison_path, dpi=150, bbox_inches=\"tight\")\nplt.show()\nprint(f\"saved comparison grid to {comparison_path}\")

## Notes / next steps for the blog post\n\n- Fill in actual observations once this has been run: did `highlight_red` preserve the rest of the pattern faithfully, or did the model redraw the whole image? Did `remove_defect` actually understand which wavy line was the defect vs. the surrounding pattern? Turbo/distilled models sometimes drift the whole image rather than making a localized edit -- worth comparing against the non-turbo `Boogu-Image-0.1-Edit` checkpoint on the same prompts if the localized-edit behavior looks weak here.\n- Try the same three prompts against one of the other synthetic-pattern defect types (`bridges`, `opens`, `vias`, `shorts`) to see if behavior is scratch-specific or general.\n- If this earns a spot in the blog post: swap in a real (rights-cleared) SEM image -- either one you have explicit permission to publish, or a genuinely open-licensed one (e.g. an image you generate yourself, or sourced from a dataset with an unambiguous CC license checked at the file level, not just the dataset's Kaggle badge).\n- `Boogu-Image-0.1-Turbo` (generation-only, ~21GB) is the natural pairing if the post also wants a from-scratch \"AI-generated defect for training-data augmentation\" angle -- not downloaded in this pass since the ask was edit-focused.